# 06 · Follow heat from value to event

## Context

Daily maximum temperature is continuous, but many impact questions concern
episodes: when heat crossed a meaningful threshold, how long it persisted, and
whether nearby locations experienced it together.

## Question

Where did multi-day heat events occur, and how closely did their occurrence
match the center of the study area?

## Analysis story

We will move through three scientific representations—value, state, and
event—then compare occurrence in space. Specialized verbs extend the same
minimal grammar rather than creating a separate workflow language.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# The two pulse groups create two heat episodes. Adding a spatial offset makes
# some pixels cross the threshold sooner or remain active longer than others.
time = pd.date_range("2025-07-01", periods=14, freq="D")
y = [40.2, 40.0, 39.8]
x = [-105.2, -105.0, -104.8]
pulse = np.array([0, 0, 5, 7, 6, 0, 0, 4, 6, 7, 5, 0, 0, 0])[:, None, None]
spatial = np.array([[-1.0, 0.0, 0.5], [-0.5, 1.0, 1.5], [-1.0, 0.5, 2.0]])[None, :, :]
cube = xr.DataArray(
    29 + pulse + spatial,
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="daily_max_temperature",
    attrs={"units": "degC"},
)
cube

## Pipes · Translate values into episodes and relationships

Each pipe names one conceptual step. Keeping them separate lets us inspect the
state Dataset before detecting events or measuring synchrony.

In [ ]:
# Which observations count as hot days?
states = (
    pipe(cube)
    | v.threshold_state(threshold=34.0, direction="above", name="hot_day")
).unwrap()

# Which hot spells persist for at least two days?
events = (
    pipe(states)
    | v.detect_events(min_duration=2, max_gap=0)
).unwrap()

# Where does hot-day occurrence match the center pixel?
synchrony = (
    pipe(states)
    | v.occurrence_synchrony(spatial_mode="reference", reference="center")
).unwrap()

# These checks document the output contracts before visualization.
assert states["state"].dtype == bool
assert len(events.catalog) > 0

## Figure · Read the progression from value to relation

The event catalog is tabular, so we count catalog rows at each pixel for a map.
That small presentation step is intentionally outside the scientific pipes.

In [ ]:
import matplotlib.pyplot as plt

# EventResult deliberately separates a cube-like Dataset from a tabular event
# catalog. Accumulate catalog rows here to make an event-count map for teaching.
event_count = xr.zeros_like(cube.isel(time=0), dtype=int)
for row in events.catalog.itertuples():
    event_count.values[row.y_index, row.x_index] += 1
sync_map = synchrony["occurrence_synchrony"].isel(time_window_end=0)

# The four panels tell the full progression: value → state → event → relation.
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
cube.mean(("y", "x")).plot(ax=axes[0, 0], marker="o", color="#8b543c")
axes[0, 0].axhline(34, color="0.3", linestyle="--", label="threshold")
axes[0, 0].legend()
axes[0, 0].set_title("Continuous regional temperature")
states["state"].mean(("y", "x")).plot(ax=axes[0, 1], marker="o", color="#3f6f72")
axes[0, 1].set_title("v.threshold_state: active fraction")
event_count.plot(ax=axes[1, 0], cmap="YlOrRd", vmin=0)
axes[1, 0].set_title("v.detect_events: events per pixel")
sync_map.plot(ax=axes[1, 1], cmap="viridis", vmin=0, vmax=1)
axes[1, 1].set_title("v.occurrence_synchrony: reference map")
plt.show()

## What the figure tells us

The regional series crosses the threshold twice. The active-fraction panel
shows that sites enter those episodes differently, the event map counts
qualifying runs, and the synchrony map shows where timing most closely matches
the center pixel.

## Try the next variation

Raise the threshold or allow a one-day gap in `v.detect_events`. Which change
alters event identity, and which changes only state classification?